In [5]:
import requests
import pandas as pd

coins = {
    'pepe': 'pepe',
    'doge': 'dogecoin',
    'shiba': 'shiba-inu'
}

all_data = []

for name, coingecko_id in coins.items():
    print(f"Descargando datos para {name.upper()}...")

    url = f'https://api.coingecko.com/api/v3/coins/{coingecko_id}/market_chart'
    params = {
        'vs_currency': 'usd',
        'days': '365',
        'interval': 'daily'
    }

    response = requests.get(url, params=params)
    data = response.json()

    # Crear DataFrames individuales
    prices_df = pd.DataFrame(data['prices'], columns=['timestamp', 'price'])
    volumes_df = pd.DataFrame(data['total_volumes'], columns=['timestamp', 'volume'])
    market_caps_df = pd.DataFrame(data['market_caps'], columns=['timestamp', 'market_cap'])

    # Convertir timestamp a fecha
    for df in [prices_df, volumes_df, market_caps_df]:
        df['date'] = pd.to_datetime(df['timestamp'], unit='ms').dt.date
        df.drop(columns='timestamp', inplace=True)

    # Combinar los tres DataFrames
    merged_df = prices_df.merge(volumes_df, on='date').merge(market_caps_df, on='date')

    # Agregar el nombre de la moneda
    merged_df['coin'] = name

    # Reordenar columnas
    merged_df = merged_df[['date', 'coin', 'price', 'volume', 'market_cap']]

    all_data.append(merged_df)

# Concatenar todo
combined_df = pd.concat(all_data, ignore_index=True)

# Guardar como CSV
combined_df.to_csv('datos_memecoins_completo.csv', index=False, sep=';')
print("✅ Archivo guardado: datos_memecoins_completo.csv")


Descargando datos para PEPE...
Descargando datos para DOGE...
Descargando datos para SHIBA...
✅ Archivo guardado: datos_memecoins_completo.csv
